# Broadcasting

The term broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations. Subject to certain constraints, the smaller array is “broadcast” across the larger array so that they have compatible shapes. Broadcasting provides a means of vectorizing array operations so that looping occurs in C instead of Python. It does this without making needless copies of data and usually leads to efficient algorithm implementations. There are, however, cases where broadcasting is a bad idea because it leads to inefficient use of memory that slows computation.

NumPy operations are usually done on pairs of arrays on an element-by-element basis. In the simplest case, the two arrays must have exactly the same shape, as in the following example:

In [1]:
import numpy as np
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 2.0, 2.0])
a * b

array([2., 4., 6.])

NumPy’s broadcasting rule relaxes this constraint when the arrays’ shapes meet certain constraints. The simplest broadcasting example occurs when an array and a scalar value are combined in an operation:

In [2]:
import numpy as np
a = np.array([1.0, 2.0, 3.0])
b = 2.0
a * b

array([2., 4., 6.])

The result is equivalent to the previous example where b was an array. We can think of the scalar b being stretched during the arithmetic operation into an array with the same shape as a. The new elements in b, as shown in Figure 1, are simply copies of the original scalar. The stretching analogy is only conceptual. NumPy is smart enough to use the original scalar value without actually making copies so that broadcasting operations are as memory and computationally efficient as possible.

The code in the second example is more efficient than that in the first because broadcasting moves less memory around during the multiplication (b is a scalar rather than an array).

## General broadcasting rules

When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing (i.e. rightmost) dimension and works its way left. Two dimensions are compatible when

### 1. they are equal, or  


### 2. one of them is 1.


### 이러면 연산가능

If these conditions are not met, a ValueError: operands could not be broadcast together exception is thrown, indicating that the arrays have incompatible shapes.

Input arrays do not need to have the same number of dimensions. The resulting array will have the same number of dimensions as the input array with the greatest number of dimensions, where the size of each dimension is the largest size of the corresponding dimension among the input arrays. Note that missing dimensions are assumed to have size one.

For example, if you have a 256x256x3 array of RGB values, and you want to scale each color in the image by a different value, you can multiply the image by a one-dimensional array with 3 values. Lining up the sizes of the trailing axes of these arrays according to the broadcast rules, shows that they are compatible:

## broadcasting은 1이라는 숫자가 확장되게 도와준다

In [3]:
a = np.array([[1,2],
              [3,4]])
b = np.array([6,7])
b2 = b.reshape(2,1)
c = np.array([8,9])

In [4]:
b2

array([[6],
       [7]])

In [5]:
a*b2

array([[ 6, 12],
       [21, 28]])

In [6]:
a*c

array([[ 8, 18],
       [24, 36]])

Image  (3d array): 256 x 256 x 3\
Scale  (1d array):             3\
Result (3d array): 256 x 256 x 3

When either of the dimensions compared is one, the other is used. In other words, dimensions with size 1 are stretched or “copied” to match the other.

In the following example, both the A and B arrays have axes with length one that are expanded to a larger size during the broadcast operation:

A      (4d array):  8 x 1 x 6 x 1\
B      (3d array):      7 x 1 x 5\
Result (4d array):  8 x 7 x 6 x 5\
-> 1 * 7 * 1 * 5

## Broadcastable arrays

A set of arrays is called “broadcastable” to the same shape if the above rules produce a valid result.

For example, if a.shape is (5,1), b.shape is (1,6), c.shape is (6,) and d.shape is () so that d is a scalar, then a, b, c, and d are all broadcastable to dimension (5,6); and

- a acts like a (5,6) array where a[:,0] is broadcast to the other columns,

- b acts like a (5,6) array where b[0,:] is broadcast to the other rows,

- c acts like a (1,6) array and therefore like a (5,6) array where c[:] is broadcast to every row, and finally,

- d acts like a (5,6) array where the single value is repeated.

Here are some more examples:

A      (2d array):  5 x 4\
B      (1d array):      1\
Result (2d array):  5 x 4

A      (2d array):  5 x 4\
B      (1d array):      4\
Result (2d array):  5 x 4

A      (3d array):  15 x 3 x 5\
B      (3d array):  15 x 1 x 5\
Result (3d array):  15 x 3 x 5

A      (3d array):  15 x 3 x 5\
B      (2d array):       3 x 5\
Result (3d array):  15 x 3 x 5

A      (3d array):  15 x 3 x 5\
B      (2d array):       3 x 1\
Result (3d array):  15 x 3 x 5

Here are examples of shapes that do not broadcast:

A      (1d array):  3\
B      (1d array):  4 # trailing dimensions do not match

A      (2d array):      2 x 1\
B      (3d array):  8 x 4 x 3 # second from last dimensions mismatched # 2가 살아남아서 안된다

In [19]:
import numpy as np
a = np.array([[ 0.0,  0.0,  0.0],
              [10.0, 10.0, 10.0],
              [20.0, 20.0, 20.0],
              [30.0, 30.0, 30.0]])
b = np.array([1.0, 2.0, 3.0])
a + b

array([[ 1.,  2.,  3.],
       [11., 12., 13.],
       [21., 22., 23.],
       [31., 32., 33.]])

what if I want to strech to column? 

In [23]:
b = np.array([1.0, 2.0, 3.0,4.0]).reshape(4,1)
a + b

array([[ 1.,  1.,  1.],
       [12., 12., 12.],
       [23., 23., 23.],
       [34., 34., 34.]])

why not working? 

Can you fix this?? 

More examples?  

Broadcasting provides a convenient way of taking the outer product (or any other outer operation) of two arrays. The following example shows an outer addition operation of two 1-d arrays:

## a[:, np.newaxis]: 1차원을 2차원으로 바꿔줌 (a: (4,) 크기였다면 (4,1)로 바꿔줌)
## np.expand_dims(a,axis = 1)도 같음 -> axs = 1은 index = 1인 자리에 새로운 축 삽입 ((4,) -> (4,1))
## axis = 0은 (4,) -> (1,4)
## reshape도 마찬가지

In [7]:
import numpy as np
a = np.array([0.0, 10.0, 20.0, 30.0]) # (4,)
b = np.array([1.0, 2.0, 3.0]) #(3,)
a[:, np.newaxis] + b # (4,1) + (3,)

array([[ 1.,  2.,  3.],
       [11., 12., 13.],
       [21., 22., 23.],
       [31., 32., 33.]])

In [8]:
a[:, np.newaxis].shape

(4, 1)

In [9]:
b.shape

(3,)

In [7]:
a[:, np.newaxis]

array([[ 0.],
       [10.],
       [20.],
       [30.]])

In [3]:
np.expand_dims(a, axis=1)

array([[ 0.],
       [10.],
       [20.],
       [30.]])

In [8]:
a.reshape(4,1)

array([[ 0.],
       [10.],
       [20.],
       [30.]])